# Credit card fraud detector

In this solution we will build the core of a credit card fraud detection system using SageMaker. We will start by training an anomaly detection algorithm, then proceed to train two XGBoost models for supervised training. To deal with the highly unbalanced data common in fraud detection, our first model will use re-weighting of the data, and the second will use re-sampling, using the popular SMOTE technique for oversampling the rare fraud data.

**Note**: When running this notebook on SageMaker Studio Domain, make sure the `Data Science` image and `Python 3` kernel are selected.

### Set up environment

In [ ]:
import sys, os, boto3, sagemaker, logging

# ── Suppress noisy botocore credential logs ───────────────────────────────
logging.getLogger('botocore').setLevel(logging.WARNING)
logging.getLogger('boto3').setLevel(logging.WARNING)
logging.getLogger('sagemaker').setLevel(logging.WARNING)

# ── SageMaker session & role ──────────────────────────────────────────────
session       = sagemaker.Session()
role          = sagemaker.get_execution_role()
region        = boto3.session.Session().region_name
bucket        = session.default_bucket()
prefix        = 'fraud-classifier'
instance_type = 'ml.m5.large'
SOLUTION_PREFIX = 'sagemaker-soln-fdml'

print(f'Role    : {role}')
print(f'Region  : {region}')
print(f'Bucket  : {bucket}')
print(f'Prefix  : {prefix}')


## Investigate and process the data

Let's start by reading in the credit card fraud data set.

In [ ]:
# Download creditcard dataset from public SageMaker solutions S3 bucket
solutions_bucket = f'sagemaker-solutions-prod-{region}'
s3_key = 'fraud-detection-using-machine-learning/data/creditcardfraud.zip'

print(f'Downloading from s3://{solutions_bucket}/{s3_key} ...')

try:
    s3 = boto3.client('s3', region_name=region)
    s3.download_file(solutions_bucket, s3_key, 'creditcardfraud.zip')
    print(f'✅ Downloaded! Size: {os.path.getsize("creditcardfraud.zip") / 1024 / 1024:.1f} MB')
except Exception as e:
    print(f'⚠️  S3 download failed: {e}')
    print('Trying alternative source...')
    import urllib.request
    url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/creditcard.csv'
    urllib.request.urlretrieve(url, 'creditcard.csv')
    print(f'✅ Downloaded creditcard.csv directly! Size: {os.path.getsize("creditcard.csv") / 1024 / 1024:.1f} MB')


In [ ]:
import os

if os.path.exists('creditcardfraud.zip'):
    from zipfile import ZipFile
    with ZipFile('creditcardfraud.zip', 'r') as zf:
        zf.extractall()
    print('✅ Extracted:', [f for f in os.listdir('.') if f.endswith('.csv')])
else:
    print('✅ creditcard.csv already present (direct download was used)')


In [ ]:
import numpy as np
import pandas as pd

data = pd.read_csv('creditcard.csv', delimiter=',')
print(f'Dataset shape: {data.shape}')


Let's take a peek at our data (we only show a subset of the columns in the table):

In [ ]:
print(data.columns.tolist())
data[['Time', 'V1', 'V2', 'V27', 'V28', 'Amount', 'Class']].describe()


The dataset contains only numerical features, because the original features have been transformed using PCA, to protect user privacy. As a result, the dataset contains 28 PCA components, V1-V28, and two features that haven't been transformed, _Amount_ and _Time_.

The class column corresponds to whether or not a transaction is fraudulent.

In [ ]:
nonfrauds, frauds = data.groupby('Class').size()
print('Number of frauds: ', frauds)
print('Number of non-frauds: ', nonfrauds)
print('Percentage of fraudulent data:', 100.*frauds/(frauds + nonfrauds))


In [ ]:
feature_columns = data.columns[:-1]
label_column    = data.columns[-1]

features = data[feature_columns].values.astype('float32')
labels   = (data[label_column].values).astype('float32')


## Training

Splitting the dataset into train and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.1, random_state=42)

print(f'Train size : {X_train.shape}')
print(f'Test size  : {X_test.shape}')


## Unsupervised Learning

We will train a Random Cut Forest anomaly detection model on the training data.

In [ ]:
from sagemaker import RandomCutForest

rcf = RandomCutForest(
    role=role,
    instance_count=1,
    instance_type=instance_type,
    data_location=f's3://{bucket}/{prefix}/',
    output_path=f's3://{bucket}/{prefix}/output',
    base_job_name=f'{SOLUTION_PREFIX}-rcf',
    num_samples_per_tree=512,
    num_trees=50
)


In [ ]:
rcf.fit(rcf.record_set(X_train))


### Host Random Cut Forest

In [ ]:
rcf_predictor = rcf.deploy(
    model_name=f'{SOLUTION_PREFIX}-rcf',
    endpoint_name=f'{SOLUTION_PREFIX}-rcf',
    initial_instance_count=1,
    instance_type=instance_type
)


In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

rcf_predictor.serializer   = CSVSerializer()
rcf_predictor.deserializer = JSONDeserializer()


### Test Random Cut Forest

In [ ]:
def predict_rcf(current_predictor, data, rows=500):
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))
    predictions = []
    for array in split_array:
        array_preds = [s['score'] for s in current_predictor.predict(array)['scores']]
        predictions.append(array_preds)
    return np.concatenate([np.array(batch) for batch in predictions])

positives        = X_test[y_test == 1]
positives_scores = predict_rcf(rcf_predictor, positives)

negatives        = X_test[y_test == 0]
negatives_scores = predict_rcf(rcf_predictor, negatives)


In [ ]:
import sys
!{sys.executable} -m pip install seaborn --quiet
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(color_codes=True)

sns.distplot(positives_scores, label='fraud',     bins=20)
sns.distplot(negatives_scores, label='not-fraud', bins=20)
plt.legend()
plt.title('RCF Anomaly Scores')
plt.show()


## Supervised Learning

### Prepare Data and Upload to S3

In [ ]:
import io, sklearn
from sklearn.datasets import dump_svmlight_file

buf = io.BytesIO()
sklearn.datasets.dump_svmlight_file(X_train, y_train, buf)
buf.seek(0)

key    = 'fraud-dataset'
subdir = 'base'
boto3.resource('s3', region_name=region).Bucket(bucket).Object(
    os.path.join(prefix, 'train', subdir, key)
).upload_fileobj(buf)

s3_train_data    = f's3://{bucket}/{prefix}/train/{subdir}/{key}'
output_location  = f's3://{bucket}/{prefix}/output'
print(f'Training data : {s3_train_data}')
print(f'Output        : {output_location}')


In [ ]:
from sagemaker.image_uris import retrieve

container = retrieve('xgboost', region, version='1.5-1')
print(f'XGBoost container: {container}')


In [ ]:
from math import sqrt

scale_pos_weight = sqrt(np.count_nonzero(y_train == 0) / np.count_nonzero(y_train))

hyperparams = {
    'max_depth'       : 5,
    'subsample'       : 0.8,
    'num_round'       : 100,
    'eta'             : 0.2,
    'gamma'           : 4,
    'min_child_weight': 6,
    'silent'          : 0,
    'objective'       : 'binary:logistic',
    'eval_metric'     : 'auc',
    'scale_pos_weight': scale_pos_weight
}
print('Hyperparameters set. scale_pos_weight =', round(scale_pos_weight, 2))


In [ ]:
clf = sagemaker.estimator.Estimator(
    container,
    role=role,
    hyperparameters=hyperparams,
    instance_count=1,
    instance_type=instance_type,
    output_path=output_location,
    sagemaker_session=session,
    base_job_name=f'{SOLUTION_PREFIX}-xgb'
)

clf.fit({'train': s3_train_data})


### Host Classifier

In [ ]:
from sagemaker.serializers import CSVSerializer

predictor = clf.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    serializer=CSVSerializer()
)
print(f'✅ Base XGBoost endpoint: {predictor.endpoint_name}')


## Evaluation

In [ ]:
def predict(current_predictor, data, rows=500):
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))
    predictions = ''
    for array in split_array:
        predictions = ','.join([predictions, current_predictor.predict(array).decode('utf-8')])
    return np.fromstring(predictions[1:], sep=',')

raw_preds = predict(predictor, X_test)


In [ ]:
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score

y_preds = np.where(raw_preds > 0.5, 1, 0)
print('Base XGBoost:')
print('  Balanced accuracy =', balanced_accuracy_score(y_test, y_preds))
print("  Cohen's Kappa     =", cohen_kappa_score(y_test, y_preds))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_predicted):
    cm      = confusion_matrix(y_true, y_predicted)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    ax = sns.heatmap(cm_norm, annot=cm, fmt='d')
    ax.set(xticklabels=['non-fraud', 'fraud'], yticklabels=['non-fraud', 'fraud'])
    ax.set_ylim([0, 2])
    plt.title('Confusion Matrix')
    plt.show()

plot_confusion_matrix(y_test, y_preds)


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_preds, target_names=['non-fraud', 'fraud']))


### SMOTE — Oversampling for Imbalanced Data

In [ ]:
import sys
!{sys.executable} -m pip install imbalanced-learn --quiet

from imblearn.over_sampling import SMOTE
from collections import Counter

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
print('Class distribution after SMOTE:', sorted(Counter(y_smote).items()))


In [ ]:
smote_buf = io.BytesIO()
sklearn.datasets.dump_svmlight_file(X_smote, y_smote, smote_buf)
smote_buf.seek(0)

key    = 'fraud-dataset-smote'
subdir = 'smote'
boto3.resource('s3', region_name=region).Bucket(bucket).Object(
    os.path.join(prefix, 'train', subdir, key)
).upload_fileobj(smote_buf)

s3_smote_train_data   = f's3://{bucket}/{prefix}/train/{subdir}/{key}'
smote_output_location = f's3://{bucket}/{prefix}/smote-output'
print(f'SMOTE training data : {s3_smote_train_data}')
print(f'Output              : {smote_output_location}')


In [ ]:
smote_hyperparams = {k: v for k, v in hyperparams.items() if k != 'scale_pos_weight'}

smote_xgb = sagemaker.estimator.Estimator(
    container,
    role=role,
    hyperparameters=smote_hyperparams,
    instance_count=1,
    instance_type=instance_type,
    output_path=smote_output_location,
    sagemaker_session=session,
    base_job_name=f'{SOLUTION_PREFIX}-xgb-smote'
)

smote_xgb.fit({'train': s3_smote_train_data})


In [ ]:
from sagemaker.serializers import CSVSerializer

smote_predictor = smote_xgb.deploy(
    initial_instance_count=1,
    model_name=f'{SOLUTION_PREFIX}-xgb-smote',
    endpoint_name=f'{SOLUTION_PREFIX}-xgb-smote',
    instance_type=instance_type,
    serializer=CSVSerializer()
)
print(f'✅ SMOTE endpoint: {smote_predictor.endpoint_name}')


In [ ]:
smote_raw_preds = predict(smote_predictor, X_test)
smote_preds     = np.where(smote_raw_preds > 0.5, 1, 0)

print('SMOTE XGBoost:')
print('  Balanced accuracy =', balanced_accuracy_score(y_test, smote_preds))
print("  Cohen's Kappa     =", cohen_kappa_score(y_test, smote_preds))


In [ ]:
plot_confusion_matrix(y_test, smote_preds)


In [ ]:
print(classification_report(y_test, smote_preds, target_names=['non-fraud', 'fraud']))


### Threshold Tuning

In [ ]:
for thres in np.linspace(0.1, 0.9, num=9):
    smote_thres_preds = np.where(smote_raw_preds > thres, 1, 0)
    print(f'Threshold: {thres:.1f}')
    print(f'  Balanced accuracy = {balanced_accuracy_score(y_test, smote_thres_preds):.3f}')
    print(f"  Cohen's Kappa     = {cohen_kappa_score(y_test, smote_thres_preds):.3f}\n")


### Production Dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve, auc

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Fraud Detection — Production Dashboard', fontsize=18, fontweight='bold', y=0.995)

# Panel 1: Score distribution
ax1 = plt.subplot(2, 3, 1)
fraud_scores = raw_preds[y_test == 1]
legit_scores = raw_preds[y_test == 0]
ax1.hist(legit_scores, bins=40, alpha=0.6, label='Legitimate', color='#1D9E75', log=True)
ax1.hist(fraud_scores, bins=40, alpha=0.6, label='Fraud',      color='#D63D2F', log=True)
ax1.axvline(x=0.5, color='black', linestyle='--', linewidth=1)
ax1.set_xlabel('XGBoost fraud probability')
ax1.set_ylabel('Count (log scale)')
ax1.legend()
ax1.set_title('Score Distribution')

# Panel 2: Confusion matrix (SMOTE)
ax2 = plt.subplot(2, 3, 2)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, smote_preds)
im = ax2.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax2.set_xticks([0,1]); ax2.set_xticklabels(['Legit','Fraud'])
ax2.set_yticks([0,1]); ax2.set_yticklabels(['Legit','Fraud'])
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')
ax2.set_title('Confusion Matrix (SMOTE)')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(cm[i,j]), ha='center', va='center',
                 color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14, fontweight='bold')

# Panel 3: Precision-Recall curve
ax3 = plt.subplot(2, 3, 3)
precision, recall, _ = precision_recall_curve(y_test, smote_raw_preds)
ax3.plot(recall, precision, color='#2563EB', linewidth=2)
ax3.set_xlabel('Recall'); ax3.set_ylabel('Precision')
ax3.set_title('Precision-Recall Curve (SMOTE)')
ax3.grid(True, alpha=0.3)

# Panel 4: Model comparison bar chart
ax4 = plt.subplot(2, 3, 4)
metrics_base  = [balanced_accuracy_score(y_test, y_preds), cohen_kappa_score(y_test, y_preds),
                 balanced_accuracy_score(y_test, y_preds), cohen_kappa_score(y_test, y_preds)]
metrics_smote = [balanced_accuracy_score(y_test, smote_preds), cohen_kappa_score(y_test, smote_preds),
                 balanced_accuracy_score(y_test, smote_preds), cohen_kappa_score(y_test, smote_preds)]
x = np.arange(2)
b1 = ax4.bar(x - 0.2, [balanced_accuracy_score(y_test, y_preds), cohen_kappa_score(y_test, y_preds)],
             0.35, label='Base XGBoost', color='#3B82F6')
b2 = ax4.bar(x + 0.2, [balanced_accuracy_score(y_test, smote_preds), cohen_kappa_score(y_test, smote_preds)],
             0.35, label='SMOTE XGBoost', color='#F97316')
ax4.set_xticks(x); ax4.set_xticklabels(['Balanced Acc', "Cohen's Kappa"])
ax4.set_ylim([0, 1.1]); ax4.legend()
ax4.set_title('Model comparison: Base vs SMOTE')

# Panel 5: Threshold tuning
ax5 = plt.subplot(2, 3, 5)
thresholds = np.linspace(0.1, 0.9, 9)
bal_accs   = [balanced_accuracy_score(y_test, np.where(smote_raw_preds > t, 1, 0)) for t in thresholds]
kappas     = [cohen_kappa_score(y_test, np.where(smote_raw_preds > t, 1, 0)) for t in thresholds]
ax5.plot(thresholds, bal_accs, 's-', color='#0EA5E9', linewidth=2, label='Balanced Acc')
ax5.plot(thresholds, kappas,   'o-', color='#EC4899', linewidth=2, label="Cohen's Kappa")
ax5.set_xlabel('Threshold'); ax5.legend()
ax5.set_title('Threshold tuning (SMOTE model)')
ax5.grid(True, alpha=0.3)

# Panel 6: Production metrics
ax6 = plt.subplot(2, 3, 6)
ax6.axis('off')
from sklearn.metrics import precision_score, recall_score, f1_score
metrics_text = [
    ('PRODUCTION METRICS', '', True),
    ('', '', False),
    (f'Total test transactions:', f'{len(y_test):,}', False),
    (f'Actual fraud cases:', f'{int(y_test.sum())}', False),
    ('', '', False),
    ('SMOTE Model:', '', True),
    (f'  Precision (fraud):', f'{precision_score(y_test, smote_preds):.0%}', False),
    (f'  Recall (fraud):', f'{recall_score(y_test, smote_preds):.0%}', False),
    (f'  F1 Score:', f'{f1_score(y_test, smote_preds):.0%}', False),
    ('', '', False),
    ('Base Model:', '', True),
    (f'  Precision (fraud):', f'{precision_score(y_test, y_preds):.0%}', False),
    (f'  Recall (fraud):', f'{recall_score(y_test, y_preds):.0%}', False),
    (f'  F1 Score:', f'{f1_score(y_test, y_preds):.0%}', False),
]
for idx, (label, value, bold) in enumerate(metrics_text):
    weight = 'bold' if bold else 'normal'
    ax6.text(0.05, 0.95 - idx*0.065, label, transform=ax6.transAxes,
             fontsize=10, fontweight=weight, verticalalignment='top', family='monospace')
    if value:
        ax6.text(0.75, 0.95 - idx*0.065, value, transform=ax6.transAxes,
                 fontsize=10, verticalalignment='top', family='monospace')

plt.tight_layout()
plt.savefig('fraud_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved as fraud_dashboard.png — ready for LinkedIn, resume, interviews')


### Deployed Endpoint Names

In [ ]:
print('RCF endpoint  :', rcf_predictor.endpoint_name)
print('Base endpoint :', predictor.endpoint_name)
print('SMOTE endpoint:', smote_predictor.endpoint_name)


### Launch Gradio Demo App

In [ ]:
!pip install gradio -q
exec(open('fraud_detection_gradio.py').read())


## Clean up

⚠️ **Run this cell only when you are completely done with the demo.**
This will delete all deployed endpoints and stop billing.

In [ ]:
# Uncomment ALL lines below only when you are done
# rcf_predictor.delete_model()
# rcf_predictor.delete_endpoint()
# predictor.delete_model()
# predictor.delete_endpoint()
# smote_predictor.delete_model()
# smote_predictor.delete_endpoint()
# print('✅ All endpoints deleted')


## Data Acknowledgements

The dataset used to demonstrate the fraud detection solution has been collected and analysed during a research collaboration of Worldline and the Machine Learning Group of ULB (Université Libre de Bruxelles) on big data mining and fraud detection.